# Lesson 1 — Simulation as a data source

*Part of the [HydrAI course](../README.md): machine-learning surrogates for physical simulation, taught on a chemical reactor.*

**Mode:** read-along (requires a kinetic mechanism — outputs shown below) · **Runtime:** if executed, ~2 min per simulation (mechanism-dependent) · **Builds on:** none — this is where the course starts

**What you'll learn**

1. Recognise when an expensive, trusted simulator is really a *training-data generator* — and why that motivates surrogate modelling.
2. Read a simulation config the way an ML practitioner should: which numbers are the input features (the knobs), which outputs become targets, and what one sample costs.
3. Follow a single plug-flow reactor (PFR) run end to end: mechanism → initial conditions → stiff ODE integration → axial profiles.
4. Explain the difference between mass fractions (`Y_*`) and mole fractions (`X_*`), and why this pipeline trains only on mass fractions.

**The example system.** HydrAI (= hydrocarbon + AI) uses one worked example throughout the course: a plug-flow reactor in which a hydrocarbon feed (n-hexane by default; ethane, propane and naphtha are also in the reactant database) flows down a heated tube and thermally cracks into lighter products. Cantera integrates the governing equations and returns temperature, pressure, velocity and species composition at 200 positions along the tube, with results exported as figures and CSV summaries. Every dataset in this course is produced exactly the way this notebook produces one run.

> ⚠️ **Read-along lesson.** This notebook needs a detailed kinetic mechanism
> that we cannot redistribute. All outputs are committed, so you can follow
> every step without running it. If you have your own Cantera mechanism, point
> `configs/simulation/main1_reactant_database.json` at it and the notebook runs
> end-to-end. Hands-on work starts in [Lesson 3](Main_3_data_exploration_feature_engineering.ipynb).

> 🧠 **Concept — Simulation as a ground-truth data source.** Supervised learning needs labelled examples, and in many scientific settings the cheapest *reliable* label generator is not an experiment but a validated simulator. A simulator gives noise-free, perfectly repeatable labels for any input you can describe; the catch is that each label has a price. The run in this notebook integrates a stiff system of coupled ODEs and costs minutes of CPU time — so anything that needs many evaluations (design optimisation, uncertainty quantification, real-time control) is off the table if every evaluation is a fresh simulation. The strategy of this course is to spend the simulation budget once, up front, to build a dataset, then train a surrogate model that reproduces the simulator's answers in microseconds. This lesson shows where one data point comes from; every later lesson is about spending that budget wisely and modelling the result honestly.
>
> *In your domain:* swap "Cantera + kinetic mechanism" for a CFD solver, a climate model, a battery electrochemistry code or an epidemic simulator — the economics, and the course, stay the same.

In [ ]:
# Setup and Imports
import sys
import os
import json
from pathlib import Path

# IMPORTANT: Import cantera BEFORE adding src to sys.path
# This prevents namespace conflict: without this, Python would find src/cantera/ 
# (our package) instead of the actual cantera library when pfr_simulator.py imports it
import cantera as ct
print(f"Cantera version: {ct.__version__}")

# Get project root: notebooks live under notebooks/; from there, project root is one level up
current_dir = Path(os.getcwd())
if (current_dir / 'src').exists():
    project_root = current_dir
elif (current_dir.parent / 'src').exists():
    project_root = current_dir.parent
else:
    project_root = current_dir
sys.path.insert(0, str(project_root))
os.chdir(project_root)  # so configs/, mechanisms/, data/ resolve correctly
print(f"Project root: {project_root}")

from src.cantera.pfr_simulator import (
    load_reactant_database,
    generate_config_for_reactant,
    setup_mechanism,
    setup_initial_conditions,
    setup_heat_flux,
    run_simulation,
    process_and_visualize_results
)

import numpy as np
import matplotlib.pyplot as plt

from src.utils.plot_style import (
    load_aesthetics,
    apply_style,
    get_profile_style,
    setup_axes,
    setup_legend,
)

figure_aesthetics = load_aesthetics()
apply_style(figure_aesthetics)

print("HydrAI: PFR Simulation System")
print("=" * 60)

## Notebook Options

User-editable flags: choose the reactant to simulate and whether to save output figures.

In [ ]:
# Reactant + I/O flags (edit configs/ml/main1_run_pfr_config.json to change these)
CONFIG_PATH = project_root / "configs" / "ml" / "main1_run_pfr_config.json"
if CONFIG_PATH.exists():
    with open(CONFIG_PATH) as _f:
        _cfg = json.load(_f)
else:
    _cfg = {}
    print(f"[WARN] Config not found: {CONFIG_PATH}. Using defaults.")

REACTANT_KEY = _cfg.get("reactant_key", "n-hexane")  # see configs/simulation/main1_reactant_database.json
IF_SAVE_PLOTS = True
PLOT_EXPORT_DIR = project_root / "outputs" / "figures" / "Main_1_run_pfr"

print(f"Reactant: {REACTANT_KEY}  |  Save plots: {IF_SAVE_PLOTS}  |  Plot dir: {PLOT_EXPORT_DIR}")

## Step 1: Select Reactant

Choose the reactant you want to simulate. Available options are listed below.

In [ ]:
# Load reactant database
database = load_reactant_database()

# Display available reactants
print("Available Reactants:")
print("-" * 60)
for key, info in database['reactants'].items():
    print(f"  {key:12s}: {info['name']:15s} - {info['description']}")
print("-" * 60)

if REACTANT_KEY not in database['reactants']:
    print(f"\n[ERROR] Reactant '{REACTANT_KEY}' not found!")
    print(f"Available reactants: {list(database['reactants'].keys())}")
else:
    reactant_info = database['reactants'][REACTANT_KEY]
    print(f"\n[OK] Selected: {reactant_info['name']} ({REACTANT_KEY})")
    print(f"    Description: {reactant_info['description']}")

## Step 2: Load Configuration

Generate the simulation configuration for the selected reactant.

In [ ]:
# Generate configuration
config = generate_config_for_reactant(REACTANT_KEY, database)
reactant_info = database['reactants'][REACTANT_KEY]

print(f"Loaded configuration for: {config['simulation_info']['title']}")
print(f"Version: {config['simulation_info']['version']}")
print(f"\nConfiguration Summary:")
print(f"  - Mechanism: {config['mechanism']['reaction_mechanism_file']}")
print(f"  - Initial Temperature: {config['initial_conditions']['temperature_K']} K")
print(f"  - Initial Pressure: {config['initial_conditions']['pressure_Pa']/1e5:.2f} bar")
print(f"  - Reactor Length: {config['reactor_geometry']['length_m']} m")
print(f"  - Reactor Diameter: {config['reactor_geometry']['diameter_m']*1000:.1f} mm")
print(f"  - Mass Flow Rate: {config['operating_conditions']['mass_flow_rate_kgps']} kg/s")
print(f"  - Number of Steps: {config['simulation_settings']['n_steps']}")

> 🧠 **Concept — The plug-flow reactor, in one paragraph.** A plug-flow reactor (PFR) is a long tube with gas flowing through it. "Plug flow" is the modelling assumption that the gas is perfectly mixed across the tube's cross-section and does not mix along it, so the only coordinate that matters is the axial distance z: a 3D vessel collapses into a 1D problem in which the state (temperature, pressure, velocity, composition) evolves as the gas travels downstream. The configuration printed above is therefore a complete recipe for one experiment: tube geometry (length, diameter), what enters it (feed species, temperature, pressure, mass flow rate), the heat supplied through the wall, and how finely to discretise the z-axis (`n_steps: 200`). That is all the reactor engineering this course requires — from here on, treat the reactor as a black-box function: six inlet/geometry knobs in, axial state profiles out.

## Step 3: Setup Mechanism and Initial Conditions

Initialize the Cantera mechanism and set up initial conditions.

In [ ]:
# Setup mechanism
print("Setting up mechanism...")
gas = setup_mechanism(config)
print(f"[OK] Mechanism loaded: {len(gas.species_names)} species, {len(gas.reaction_equations())} reactions")

# Setup initial conditions
print("\nSetting up initial conditions...")
T_0, p_0, composition_0, length, diameter, area, roughness, mass_flow_rate, u_0 = \
    setup_initial_conditions(gas, config)

print(f"[OK] Initial conditions set:")
print(f"  - Temperature: {T_0:.1f} K")
print(f"  - Pressure: {p_0/1e5:.2f} bar")
print(f"  - Initial velocity: {u_0:.2f} m/s")
print(f"  - Composition: {composition_0}")

# Setup heat flux
print("\nSetting up heat flux profile...")
hf, z_profile, heatflux_profile = setup_heat_flux(config)
print(f"[OK] Heat flux profile loaded: {len(heatflux_profile)} data points")
print(f"  - Average heat flux: {np.mean(heatflux_profile)/1e3:.1f} kW/m²")

## Step 4: Run Simulation

Execute the PFR simulation. This may take a few minutes depending on the mechanism complexity.

> 🧠 **Concept — Why this is the expensive cell: stiff ODE integration.** Marching down the tube means solving a coupled ODE system: one equation per chemical species plus mass, momentum and energy balances — hundreds of equations for a detailed mechanism. Cracking chemistry makes the system *stiff*: radical intermediates react on microsecond timescales while the bulk flow evolves over seconds, so the same system contains timescales separated by many orders of magnitude. An explicit solver would need absurdly small steps to remain stable, so implicit solvers (SUNDIALS' CVODES, under Cantera's hood) are used instead, and each step involves building and factorising a Jacobian — which is why cost climbs steeply with mechanism size. The number to remember is the wall-clock time printed by the next cell: that is the price of *one* labelled example, for one set of inlet conditions. When later lessons report surrogate inference in microseconds, this is the ~10⁶× speed-up being bought — and the data generated this way is what it costs.

In [ ]:
# Run simulation
print("=" * 60)
print("Starting PFR Simulation...")
print("=" * 60)

import time
start_time = time.time()

states1 = run_simulation(gas, config, reactant_info, hf, T_0, p_0, 
                        length, diameter, area, roughness, mass_flow_rate, u_0)

elapsed_time = time.time() - start_time
print(f"\n[OK] Simulation completed in {elapsed_time:.2f} seconds")
print(f"  - Number of points: {len(states1.z)}")
print(f"  - Final temperature: {states1.T[-1]:.1f} K")
print(f"  - Final pressure: {states1.P[-1]/1e5:.2f} bar")

#### 🤔 Think it through — the cost of one label

The timing printed above is the cost of a single (inlet conditions → profiles) evaluation. Suppose you want to *optimise* this reactor — find the inlet conditions that maximise the yield of a valuable product — with an optimiser that needs ~10,000 objective evaluations. What does that cost if every evaluation is a fresh simulation, and what are your options?

<details><summary><b>💡 Discussion</b></summary>

At roughly two minutes per run, 10,000 evaluations is on the order of two CPU-weeks — for one optimisation, of one objective, before any refinement. You could parallelise (helps, but optimisers are partly sequential: each proposal depends on previous results), or simplify the mechanism (cheaper but less trustworthy labels), or do what this course does: run an affordable, well-designed batch of simulations once (Lesson 2), train a surrogate on it (Lessons 4–8), and let the optimiser hammer the surrogate at microseconds per call (Lesson 10) — validating only the final candidate against the real simulator. The optimisation itself then costs seconds; the honest open question, which the rest of the course takes seriously, is whether the surrogate can be trusted in the region the optimiser is drawn to.
</details>

## Step 5: Process Results and Visualize

Generate visualizations and calculate key performance metrics.

In [ ]:
# Process and visualize results
print("Processing results and generating visualizations...")
conversion, yields = process_and_visualize_results(gas, states1, config, 
                                                  reactant_info, hf, T_0, p_0, u_0)

print("[OK] Results processed and visualizations generated")

## Step 6: Results Summary

Display key simulation results and performance metrics.

In [ ]:
# Calculate residence time
# numpy >=2.0 renamed trapz -> trapezoid and removed trapz outright in >=2.4;
# a ternary picks the right one lazily.
_trapz = np.trapezoid if hasattr(np, "trapezoid") else np.trapz
residence_time = _trapz(1/states1.velocity, states1.z)

# Display summary
print("=" * 60)
print("SIMULATION COMPLETED SUCCESSFULLY!")
print("=" * 60)
print(f"[OK] {reactant_info['name']} conversion: {conversion:.1f}%")
print(f"[OK] Temperature rise: {states1.T[-1] - T_0:.1f} K")
print(f"[OK] Pressure drop: {(p_0 - states1.P[-1])/1e5:.2f} bar")
print(f"[OK] Residence time: {residence_time:.3f} s")
print(f"
[OK] Product Yields:")
for product, yield_val in yields.items():
    print(f"  - {product}: {yield_val:.2f}%")
print(f"
[OK] Files generated:")
print(f"  - outputs/figures/*.png: Visualization plots")
print(f"  - outputs/results/*.csv: Simulation data")
print(f"  - outputs/results/*.dat: Simulation summary")
print("=" * 60)

#### 🤔 Think it through — why keep the whole profile?

The summary above condenses this run to a handful of exit numbers: conversion, exit temperature, yields. Yet the pipeline stores the state at all 200 axial positions of every run — 200× more data. Why is that worth it?

<details><summary><b>💡 Discussion</b></summary>

Three reasons. First, data efficiency: the simulation is the expensive part, and once it has run, the 199 intermediate states are free — each run yields 200 training rows (inputs + z → state) instead of one. Second, capability: an exit-only model answers exactly one question, while a full-profile model (Lessons 5–7) predicts the state as a function of position — needed to see *where* in the reactor things happen, and a prerequisite for Lesson 7's physics-informed network, which penalises ODE residuals along z (you cannot differentiate a profile you never predicted). Third, quality control: unphysical behaviour is far easier to spot in a profile than in a single exit value. The catch: the 200 rows of one run are strongly correlated, so they must never be scattered randomly across train and test sets — that is why every split later in the course is made at *run* level.
</details>

## Quick Visualization

Display key profiles inline in the notebook.

> 🧠 **Concept — Mass fractions (Y) vs mole fractions (X).** Chemical composition can be expressed per unit mass (mass fractions, `Y`) or per molecule count (mole fractions, `X`). Both sum to one, but they behave very differently as ML targets. Cracking one hexane molecule into two or three smaller molecules changes the total *number* of molecules in the gas, so every species' mole fraction shifts — even species that took no part in the reaction. Mass, by contrast, is conserved: mass fractions redistribute only among the species actually produced or consumed. That gives `Y` a clean, machine-checkable structure (ΣY = 1, with each element's mass conserved) that later lessons exploit both as a data-quality check and as a physics term in the PINN loss. This is why the entire HydrAI pipeline standardises on `Y_*` columns and never trains on `X_*`. The conversion profile plotted below uses the same currency: the feed's mass fraction relative to its inlet value.

In [ ]:
# Quick inline visualization (sizes, colors, grid from configs/style/figure_aesthetics.json)
w, h = tuple(figure_aesthetics['figure']['figsize'])
fig, axes = plt.subplots(2, 2, figsize=(w * 1.8, h * 10 / 6))

# Temperature profile
_st = get_profile_style('temperature', figure_aesthetics)
axes[0, 0].plot(
    states1.z, states1.T,
    color=_st['color'], linewidth=_st['linewidth'],
    linestyle=_st['linestyle'], alpha=_st['alpha'],
)
axes[0, 0].set_xlabel('Reactor Position [m]')
axes[0, 0].set_ylabel(_st['ylabel'])
axes[0, 0].set_title(_st['title'])
setup_axes(axes[0, 0], figure_aesthetics)

# Pressure profile
_st = get_profile_style('pressure', figure_aesthetics)
axes[0, 1].plot(
    states1.z, states1.P / 1e5,
    color=_st['color'], linewidth=_st['linewidth'],
    linestyle=_st['linestyle'], alpha=_st['alpha'],
)
axes[0, 1].set_xlabel('Reactor Position [m]')
axes[0, 1].set_ylabel(_st['ylabel'])
axes[0, 1].set_title(_st['title'])
setup_axes(axes[0, 1], figure_aesthetics)

# Velocity profile
_st = get_profile_style('velocity', figure_aesthetics)
axes[1, 0].plot(
    states1.z, states1.velocity,
    color=_st['color'], linewidth=_st['linewidth'],
    linestyle=_st['linestyle'], alpha=_st['alpha'],
)
axes[1, 0].set_xlabel('Reactor Position [m]')
axes[1, 0].set_ylabel(_st['ylabel'])
axes[1, 0].set_title(_st['title'])
setup_axes(axes[1, 0], figure_aesthetics)

# Reactant conversion
# Get species index - handle species name format (may have :1 suffix)
feed_species = reactant_info['feed_species']
# Remove :1 suffix if present (format from database vs mechanism)
species_name = feed_species.split(':')[0] if ':' in feed_species else feed_species

try:
    species_idx = gas.species_index(species_name)
except:
    # Fallback: search for species by partial match
    species_idx = None
    for i, species in enumerate(gas.species_names):
        if species_name in species or reactant_info['name'].lower() in species.lower():
            species_idx = i
            break
    if species_idx is None:
        raise ValueError(f"Could not find species: {feed_species} (tried: {species_name})")

# Access mass fractions from SolutionArray using 2D indexing: states1.Y[:, species_idx]
reactant_mass_frac = states1.Y[:, species_idx]
conversion_profile = (1 - reactant_mass_frac / reactant_mass_frac[0]) * 100
_st = get_profile_style('conversion', figure_aesthetics)
axes[1, 1].plot(
    states1.z, conversion_profile,
    color=_st['color'], linewidth=_st['linewidth'],
    linestyle=_st['linestyle'], alpha=_st['alpha'],
)
axes[1, 1].set_xlabel('Reactor Position [m]')
axes[1, 1].set_ylabel('Conversion [%]')
axes[1, 1].set_title(f'{reactant_info["name"]} Conversion Profile')
setup_axes(axes[1, 1], figure_aesthetics)

plt.tight_layout()
if IF_SAVE_PLOTS:
    PLOT_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(PLOT_EXPORT_DIR / f"main1_quick_profiles_{REACTANT_KEY}.png", dpi=150, bbox_inches='tight')
plt.show()

print("\n[OK] Inline visualizations displayed above")

In [ ]:
# Combined Reactant Conversion and Product Formation
font_cfg = figure_aesthetics['font']
w, h = tuple(figure_aesthetics['figure']['figsize'])
fig, ax = plt.subplots(figsize=(w * 1.2, h * 7 / 6))

# Get reactant conversion profile
feed_species = reactant_info['feed_species']
species_name = feed_species.split(':')[0] if ':' in feed_species else feed_species

try:
    species_idx = gas.species_index(species_name)
except:
    # Fallback: search for species by partial match
    species_idx = None
    for i, species in enumerate(gas.species_names):
        if species_name in species or reactant_info['name'].lower() in species.lower():
            species_idx = i
            break
    if species_idx is None:
        raise ValueError(f"Could not find species: {feed_species}")

# Calculate reactant conversion
reactant_mass_frac = states1.Y[:, species_idx]
conversion_profile = (1 - reactant_mass_frac / reactant_mass_frac[0]) * 100

# Plot reactant conversion (increasing) or reactant remaining (decreasing)
# Option 1: Show conversion (0% to 100% as reactant is consumed)
conversion_profile = (1 - reactant_mass_frac / reactant_mass_frac[0]) * 100
# Option 2: Show reactant remaining (100% to 0% as reactant is consumed)
reactant_remaining = (reactant_mass_frac / reactant_mass_frac[0]) * 100

_st_rem = get_profile_style('temperature', figure_aesthetics)
prod_style = get_profile_style('products', figure_aesthetics)
prod_colors = prod_style.get('colors') or [prod_style['color']]

# Plot reactant remaining (decreasing) - this shows reactant disappearing
ax.plot(
    states1.z, reactant_remaining,
    color=_st_rem['color'], linewidth=_st_rem['linewidth'],
    linestyle=_st_rem['linestyle'], alpha=_st_rem['alpha'],
    label=f'{reactant_info["name"]} Remaining',
)

# Get target products and plot their formation
target_products = reactant_info.get('target_products', [])
product_names = reactant_info.get('product_names', [])

# Calculate and plot product yields (as percentage of initial reactant)
initial_reactant_mass = reactant_mass_frac[0]
_ln = figure_aesthetics['line']
for i, product in enumerate(target_products):
    try:
        # Handle product name format (may have :1 suffix)
        product_name_clean = product.split(':')[0] if ':' in product else product
        product_idx = gas.species_index(product_name_clean)

        # Get product mass fraction profile
        product_mass_frac = states1.Y[:, product_idx]

        # Calculate product yield as percentage of initial reactant
        # Product yield = (product mass fraction / initial reactant mass fraction) * 100
        # This shows how much product is formed relative to initial reactant
        product_yield = (product_mass_frac / initial_reactant_mass) * 100

        product_name = product_names[i] if i < len(product_names) else product_name_clean
        ax.plot(
            states1.z, product_yield,
            color=prod_colors[i % len(prod_colors)],
            linewidth=_ln['width'],
            linestyle=_ln['style'],
            alpha=_ln['alpha'],
            label=product_name,
        )
    except:
        continue

ax.set_xlabel('Reactor Position [m]')
ax.set_ylabel('Reactant Remaining / Product Yield [%]')
ax.set_title(
    f'{reactant_info["name"]} Consumption and Products Formation',
    fontsize=font_cfg['title_size'],
    fontweight=font_cfg['title_weight'],
)
setup_legend(ax, figure_aesthetics, ncol=4)
setup_axes(ax, figure_aesthetics)
ax.set_ylim(bottom=0)

plt.tight_layout()
if IF_SAVE_PLOTS:
    PLOT_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(PLOT_EXPORT_DIR / f"main1_conversion_products_{REACTANT_KEY}.png", dpi=150, bbox_inches='tight')
plt.show()

print("\n[OK] Combined conversion and product formation plot displayed above")

#### 🤔 Think it through — is this really "ground truth"?

Throughout this course, simulation output is treated as ground truth for the surrogate models. A physical experiment would give noisy but real data; the simulator gives noise-free but model-dependent data. Which kinds of error can a surrogate trained on this data never escape, and which can it actually control?

<details><summary><b>💡 Discussion</b></summary>

A surrogate can only be as right as its teacher. Whatever the kinetic mechanism gets wrong — missing reaction pathways, imperfect rate constants, plus the plug-flow idealisation itself (no radial gradients, no wall effects) — is inherited wholesale, and it is invisible to every metric in this course, because validation is computed against the same simulator. What the surrogate *can* control is the error it adds on top: interpolation error inside the sampled domain and extrapolation error outside it. Keep the two comparisons separate in your head: surrogate-vs-simulator (Lessons 4–9; measurable, and the subject of this course) and simulator-vs-reality (a modelling and validation question outside our scope — but the one that matters most the day someone acts on a prediction).
</details>